# Project for Deep learning course

In [1]:
# importing dependencies and downloading dataset

!uv sync

import datasets as d

dataset = d.load_dataset("fancyzhx/amazon_polarity")

Resolved 275 packages in 1ms
Checked 270 packages in 3ms


In [2]:
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'title', 'content'],
        num_rows: 3600000
    })
    test: Dataset({
        features: ['label', 'title', 'content'],
        num_rows: 400000
    })
})

Downloading the generative model:

In [3]:
from transformers import pipeline

MODEL_NAME = 'HuggingFaceTB/SmolLM-135M-Instruct'

pipe = pipeline('text-generation', 
                model=MODEL_NAME, 
                device_map='auto', 
                dtype='auto')

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [ ]:
prompt =    'If youve played the game, you know how divine the music is! Every single song tells a story of the game, its that good! The greatest songs are without a doubt, Chrono Cross: Times Scar, Magical Dreamers: The Wind, The Stars, and the Sea and Radical Dreamers: Unstolen Jewel. (Translation varies) This music is perfect if you ask me, the best it can be. Yasunori Mitsuda just poured his heart on and wrote it down on paper. => positive' \
            'This was a easy to read book that made me want to keep reading on and on, not easy to put down.It left me wanting to read the follow on, which I hope is coming soon. I used to read a lot but have gotten away from it. This book made me want to read again. Very enjoyable. => positive' \
            'Its glaringly obvious that all of the glowing reviews have been written by the same person, perhaps the author herself. They all have the same misspellings and poor sentence structure that is featured in the book. Who made Veronica Haddon think she is an author? => negative' \
            'sizes are much smaller than what is recomended in the chart. I tried to put it and sheer it!. I guess you should not buy this item in the internet..it is better to go to the store and check it => ;'


In [5]:
print(pipe(prompt, max_length=25, max_new_tokens=512))

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=25) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': "If youve played the game, you know how divine the music is! Every single song tells a story of the game, its that good! The greatest songs are without a doubt, Chrono Cross: Times Scar, Magical Dreamers: The Wind, The Stars, and the Sea and Radical Dreamers: Unstolen Jewel. (Translation varies) This music is perfect if you ask me, the best it can be. Yasunori Mitsuda just poured his heart on and wrote it down on paper. => positive;This was a easy to read book that made me want to keep reading on and on, not easy to put down.It left me wanting to read the follow on, which I hope is coming soon. I used to read a lot but have gotten away from it. This book made me want to read again. Very enjoyable. => positive;Its glaringly obvious that all of the glowing reviews have been written by the same person, perhaps the author herself. They all have the same misspellings and poor sentence structure that is featured in the book. Who made Veronica Haddon think she is an author

## Hypertuning model

dataset prep

In [6]:
from pprint import PrettyPrinter
pprint = PrettyPrinter(compact=True).pprint

dataset = dataset.shuffle()
pprint(dataset["train"][0])

{'content': 'I work at a retail carpet store. My Zircon has come in very handy '
            'when I have to measure at homes where the owner was too lazy to '
            'pick up clothes, toys, etc. off the floor. I no longer have to '
            'pick up dirty socks or underwear (yes, underwear) to do a correct '
            "measure. It's VERY accurate.",
 'label': 1,
 'title': 'Zircon laser'}


In [7]:
import transformers

model_name_gen = "HuggingFaceTB/SmolLM-135M-Instruct"
model_name_bi = "bert-base-cased"
tokenizer_gen = transformers.AutoTokenizer.from_pretrained(model_name_gen)
tokenizer_bi = transformers.AutoTokenizer.from_pretrained(model_name_bi)

In [8]:
# Define a simple function that applies the tokenizer
def tokenize(example):
    return tokenizer_gen(
        example["content"],
        max_length=128,
        truncation=True,
    )

# Apply the tokenizer to the whole dataset using .map()
dataset_gen = dataset.map(tokenize)

Map:   0%|          | 0/3600000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400000 [00:00<?, ? examples/s]

In [9]:
# Define a simple function that applies the tokenizer
def tokenize(example):
    return tokenizer_bi(
        example["content"],
        max_length=128,
        truncation=True,
    )

# Apply the tokenizer to the whole dataset using .map()
dataset_bi = dataset.map(tokenize)

Map:   0%|          | 0/3600000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400000 [00:00<?, ? examples/s]

### Model prep

In [10]:
import torch


# This gives a new name to the config class, just for convenience
BasicConfig = transformers.PretrainedConfig


# This is the model
class SimpleCNN(transformers.PreTrainedModel):

    config_class = BasicConfig

    # In the initialization method, one instantiates the layers
    # these will be the parameters of the model
    def __init__(self, config):
        super().__init__(config)
        # Embedding layer: vocab size x embedding dim
        self.embeddings = torch.nn.Embedding(
            num_embeddings=config.vocab_size,
            embedding_dim=config.embedding_dim
        )
        # Convolution layer
        self.convolution = torch.nn.Conv1d(
            config.embedding_dim,
            config.num_filters,
            config.filter_size,
            padding=1
        )
        # Activation function following convolution
        self.activation = torch.nn.ReLU()
        # Pooling layer: global max pooling, regardless of input length
        self.pooling_layer = torch.nn.AdaptiveMaxPool1d(
            output_size=1
        )
        # Output layer: num filters to output size
        self.output_layer = torch.nn.Linear(
            in_features=config.num_filters,
            out_features=config.num_labels
        )
        # Loss function: standard loss for classification
        self.loss = torch.nn.CrossEntropyLoss()

    def forward(self, input_ids, labels=None, attention_mask=None):
        #shape of input: [batch_size, maxlen]
        x = self.embeddings(input_ids)
        #shape of x: [batch_size, maxlen, embedding_dim]
        x = x.permute((0,2,1))
        #shape of x: [batch_size, embedding_dim, maxlen]
        x = self.convolution(x)
        #shape of x: [batch_size, filters, maxlen]
        x = self.activation(x)
        #shape of x: [batch_size, filters, maxlen]
        x = self.pooling_layer(x)
        #shape of x: [batch_size, filters, 1]
        x = x.flatten(start_dim=1)
        #shape of x: [batch_size, filters]
        output = self.output_layer(x)

        # Return value computed as in the MLP:
        if labels is not None:
            # We have labels, so we can calculate the loss
            return (self.loss(output,labels), output)
        else:
            # No labels, so just return the output
            return (output,)

In [11]:
config_gen = BasicConfig(
    vocab_size = tokenizer_gen.vocab_size,
    num_labels = len(set(dataset_gen['train']['label'])),
    embedding_dim = 64,
    filter_size = 3,
    num_filters = 10,
)

model_gen = SimpleCNN(config_gen)

In [12]:
config_bi = BasicConfig(
    vocab_size = tokenizer_bi.vocab_size,
    num_labels = len(set(dataset_bi['train']['label'])),
    embedding_dim = 64,
    filter_size = 3,
    num_filters = 10,
)

model_bi = SimpleCNN(config_bi)

In [28]:
# Set training arguments
trainer_args = transformers.TrainingArguments(
    "checkpoints",
    eval_strategy="steps",
    logging_strategy="steps",
    load_best_model_at_end=True,
    eval_steps=500,
    logging_steps=1000,
    learning_rate=0.003,
    per_device_train_batch_size=128,
    max_steps=10000,
    report_to="none",
)

In [14]:
import evaluate

accuracy = evaluate.load("accuracy")


def compute_accuracy(outputs_and_labels):
    outputs, labels = outputs_and_labels
    predictions = outputs.argmax(axis=-1) #pick the index of the "winning" label
    return accuracy.compute(predictions=predictions, references=labels)

data_collator_gen = transformers.DataCollatorWithPadding(tokenizer_gen)
data_collator_bi = transformers.DataCollatorWithPadding(tokenizer_bi)

# Argument gives the number of steps of patience before early stopping
early_stopping = transformers.EarlyStoppingCallback(
    early_stopping_patience=5
)

In [25]:
from collections import defaultdict

class LogSavingCallback(transformers.TrainerCallback):
    def on_train_begin(self, *args, **kwargs):
        self.logs = defaultdict(list)
        self.training = True

    def on_train_end(self, *args, **kwargs):
        self.training = False

    def on_log(self, args, state, control, logs, model=None, **kwargs):
        if self.training:
            for k, v in logs.items():
                if k != "epoch" or v not in self.logs[k]:
                    self.logs[k].append(v)

training_logs = LogSavingCallback()

| Values | Training loss | Validation loss | Accuracy |
| :- | :- | :- | :- |
| eval_steps=500, logging_steps=1000, learning_rate=0.005, per_device_train_batch_size=32, max_steps=5000 | 0.332245 | 0.335693 | 0.853193 |
| eval_steps=500, logging_steps=1000, learning_rate=0.004, per_device_train_batch_size=32, max_steps=5000 | 0.302831 | 0.316371 | 0.864190 |
| eval_steps=500, logging_steps=1000, learning_rate=0.004, per_device_train_batch_size=64, max_steps=5000 | 0.296319 | 0.293647 | 0.874965 |
| eval_steps=500, logging_steps=1000, learning_rate=0.003, per_device_train_batch_size=64, max_steps=5000 | 0.288178 | 0.292558 | 0.876300 |
| eval_steps=500, logging_steps=1000, learning_rate=0.003, per_device_train_batch_size=128, max_steps=5000 | 0.279274 | 0.279443 | 0.882840 |
| eval_steps=500, logging_steps=1000, learning_rate=0.003, per_device_train_batch_size=128, max_steps=10000 | 0.264760 | 0.264431 | 0.890630 |

In [29]:
trainer_gen = transformers.Trainer(
    model=model_gen,
    args=trainer_args,
    train_dataset=dataset_gen["train"],
    eval_dataset=dataset_gen["test"],
    compute_metrics=compute_accuracy,
    data_collator=data_collator_gen,
    callbacks=[early_stopping, training_logs]
)

trainer_gen.train()

/home/lauris/Documents/school/2026-27/Deep-Learning/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Accuracy
500,No log,0.350714,0.872295
1000,0.152799,0.358938,0.868120
1500,0.152799,0.321579,0.870780
2000,0.229134,0.303493,0.874290
2500,0.229134,0.288743,0.878607
3000,0.266781,0.286801,0.880420
3500,0.266781,0.279930,0.882567
4000,0.270803,0.280696,0.882675
4500,0.270803,0.285239,0.879850
5000,0.274101,0.275037,0.885100


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/lauris/Documents/school/2026-27/Deep-Learning/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/lauris/Documents/school/2026-27/Deep-Learning/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


... (32 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/lauris/Documents/school/2026-27/Deep-Learning/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=10000, training_loss=0.2535586944580078, metrics={'train_runtime': 1125.8459, 'train_samples_per_second': 1136.923, 'train_steps_per_second': 8.882, 'total_flos': 1918894080000.0, 'train_loss': 0.2535586944580078, 'epoch': 0.35555555555555557})

| Values | Training loss | Validation loss | Accuracy |
| :- | :- | :- | :- |
| eval_steps=500, logging_steps=1000, learning_rate=0.005, per_device_train_batch_size=32, max_steps=5000 | 0.332518 | 0.337723 | 0.852158 |
| eval_steps=500, logging_steps=1000, learning_rate=0.003, per_device_train_batch_size=128, max_steps=10000 | 0.275328 | 0.274361 | 0.885470 |

In [30]:
trainer_bi = transformers.Trainer(
    model=model_bi,
    args=trainer_args,
    train_dataset=dataset_bi["train"],
    eval_dataset=dataset_bi["test"],
    compute_metrics=compute_accuracy,
    data_collator=data_collator_bi,
    callbacks=[early_stopping, training_logs]
)

trainer_bi.train()

/home/lauris/Documents/school/2026-27/Deep-Learning/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Accuracy
500,No log,0.339067,0.853753
1000,0.296375,0.331741,0.858320
1500,0.296375,0.314205,0.865367
2000,0.312444,0.311680,0.866100
2500,0.312444,0.303778,0.870140
3000,0.305532,0.298870,0.873370
3500,0.305532,0.293514,0.875938
4000,0.295655,0.297991,0.873310
4500,0.295655,0.295920,0.874123
5000,0.287324,0.286739,0.879138


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/lauris/Documents/school/2026-27/Deep-Learning/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/lauris/Documents/school/2026-27/Deep-Learning/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


... (32 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/lauris/Documents/school/2026-27/Deep-Learning/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=10000, training_loss=0.28945531311035155, metrics={'train_runtime': 1095.0368, 'train_samples_per_second': 1168.911, 'train_steps_per_second': 9.132, 'total_flos': 1918894080000.0, 'train_loss': 0.28945531311035155, 'epoch': 0.35555555555555557})